# CSI 300 parallel minute-cache generator

Read raw market pickle files, build market partitions, merge one complete `basket_tick_v03` cache per trading day, and optionally remove the partitions after validation.


In [29]:
import importlib
from pathlib import Path
import sys

_root_candidates = [Path.cwd().resolve(), Path.cwd().resolve() / "Stock-Index-Fitting"]
PROJECT_ROOT = next(
    (candidate for candidate in _root_candidates if (candidate / "utils" / "cache_generator.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Cannot locate Stock-Index-Fitting from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
from xtquant import xtdata

import utils.minute_tick_cache_v03 as minute_tick_cache_module
minute_tick_cache_module = importlib.reload(minute_tick_cache_module)
import utils.cache_generator as cache_generator_module
cache_generator_module = importlib.reload(cache_generator_module)

from utils.cache_generator import (
    CSI300_INDEX_CODE,
    generate_csi300_caches,
    merge_partition_caches_for_range,
    preview_generation_plan,
    resolve_trading_dates,
    select_complete_trade_dates,
)


## Configuration

`START_DATE` and `END_DATE` control both partition generation and complete-cache merging. Non-trading endpoints are moved to the closest previous SH trading day through the XtQuant calendar.


In [30]:
INDEX_CODE = CSI300_INDEX_CODE  # Fixed to CSI 300: 000300
START_DATE = "20260701"
END_DATE = "20260730"
XT_PORT = 58610
MAX_WORKERS = 4
FORCE_REBUILD = False

# Delete validated market partition pickle files after a complete cache exists.
DELETE_PARTITION_CACHES = True

# Rebuild an already valid complete cache during the merge phase.
OVERWRITE_COMPLETE_CACHE = False

SOURCE_TICK_ROOT = Path(r"\\192.168.1.138\康曼德共享\高频行情迅投\ticks")
WEIGHTS_DIR = PROJECT_ROOT / "data" / "weights_projection"
CACHE_DIR = PROJECT_ROOT / "data" / INDEX_CODE / "_tick_cache_correlation_v03"

print("Project root:", PROJECT_ROOT)
print("Source tick root:", SOURCE_TICK_ROOT)
print("Weights directory:", WEIGHTS_DIR)
print("Cache directory:", CACHE_DIR)
print("Workers:", MAX_WORKERS)
print("Force rebuild:", FORCE_REBUILD)
print("Delete partition caches:", DELETE_PARTITION_CACHES)


Project root: E:\Codex\系统\Stock-Index-Fitting
Source tick root: \\192.168.1.138\康曼德共享\高频行情迅投\ticks
Weights directory: E:\Codex\系统\Stock-Index-Fitting\data\weights_projection
Cache directory: E:\Codex\系统\Stock-Index-Fitting\data\000300\_tick_cache_correlation_v03
Workers: 4
Force rebuild: False
Delete partition caches: True


## Resolve trading dates


In [31]:
xtdata.reconnect(port=XT_PORT)
date_resolution = resolve_trading_dates(
    START_DATE,
    END_DATE,
    xtdata_client=xtdata,
)
trade_dates = list(date_resolution.trade_dates)

print("Configured range:", date_resolution.configured_start_date, date_resolution.configured_end_date)
print("Adjusted range:  ", date_resolution.adjusted_start_date, date_resolution.adjusted_end_date)
print(f"Trading dates ({len(trade_dates)}):", trade_dates)


***** xtdata连接成功 2026-07-31 14:15:26*****
服务信息: {'tag': 'qmt_research', 'version': '1.0'}
服务地址: 127.0.0.1:58610
数据路径: E:\迅投极速交易终端睿智融科版\datadir
设置xtdata.enable_hello = False可隐藏此消息

Configured range: 20260701 20260730
Adjusted range:   20260701 20260730
Trading dates (22): ['20260701', '20260702', '20260703', '20260706', '20260707', '20260708', '20260709', '20260710', '20260713', '20260714', '20260715', '20260716', '20260717', '20260720', '20260721', '20260722', '20260723', '20260724', '20260727', '20260728', '20260729', '20260730']


## Preview inputs

This metadata-only check does not deserialize the large source pickle files.


In [32]:
plan = preview_generation_plan(
    trade_dates,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
)
display(plan)

preview_complete_trade_dates, excluded_trade_date_reasons = select_complete_trade_dates(
    plan,
    trade_dates,
)

if excluded_trade_date_reasons:
    print("\nExcluded incomplete dates:")
    for excluded_date, reasons in excluded_trade_date_reasons.items():
        print(f"  {excluded_date}")
        for reason in reasons:
            print(f"    - {reason}")

print(f"\nDates with complete preview inputs ({len(preview_complete_trade_dates)}):", preview_complete_trade_dates)
print(f"Dates scheduled for cache generation or unavailable marking ({len(trade_dates)}):", trade_dates)
if not trade_dates:
    raise RuntimeError("No trading dates are available for cache generation.")


,trade_date,weight_file,component_count,market,market_stock_count,source_path,source_exists,source_gb,final_cache_path,final_cache_exists
0,20260701,沪深300_样本权重_20260630.csv,300,sh_kcb,20,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,0.746,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
1,20260701,沪深300_样本权重_20260630.csv,300,sh_zb,169,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,2.119,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
2,20260701,沪深300_样本权重_20260630.csv,300,sz_cyb,34,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\...,True,1.618,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
3,20260701,沪深300_样本权重_20260630.csv,300,sz_zb,77,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\...,True,1.752,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
4,20260702,沪深300_样本权重_20260630.csv,300,sh_kcb,20,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,0.740,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
...,...,...,...,...,...,...,...,...,...,...
83,20260729,沪深300_样本权重_20260630.csv,300,sz_zb,77,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\...,False,NaN,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
84,20260730,沪深300_样本权重_20260630.csv,300,sh_kcb,20,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,0.720,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
85,20260730,沪深300_样本权重_20260630.csv,300,sh_zb,169,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\...,True,2.150,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
86,20260730,沪深300_样本权重_20260630.csv,300,sz_cyb,34,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\...,True,1.593,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False



Excluded incomplete dates:
  20260720
    - Missing source pickle: \\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\20260720_tick_sh_kcb.pkl
    - Missing source pickle: \\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\07\20260720_tick_sh_zb.pkl
    - Missing source pickle: \\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\20260720_tick_sz_cyb.pkl
    - Missing source pickle: \\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\20260720_tick_sz_zb.pkl
  20260729
    - Missing source pickle: \\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\20260729_tick_sz_cyb.pkl
    - Missing source pickle: \\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\07\20260729_tick_sz_zb.pkl

Dates with complete preview inputs (20): ['20260701', '20260702', '20260703', '20260706', '20260707', '20260708', '20260709', '20260710', '20260713', '20260714', '20260715', '20260716', '20260717', '20260721', '20260722', '20260723', '20260724', '20260727', '20260728', '20260730']
Dates scheduled for cache generation or unavailable marking (22): ['20

## Generate market partitions and complete caches

Dates are processed sequentially. Physical market files within one date are handled by the configured process pool.


In [33]:
generated_cache_paths = generate_csi300_caches(
    trade_dates,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
    max_workers=MAX_WORKERS,
    force_rebuild=FORCE_REBUILD,
)

print(f"Complete caches available after generation: {len(generated_cache_paths)}")
for cache_path in generated_cache_paths:
    print(" ", cache_path)


[DATE 1/22] 20260701: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260630.csv, components=300
  [partition:sh_kcb] built (29.2s)
  [partition:sz_cyb] built (55.5s)
  [partition:sz_zb] built (58.7s)
  [partition:sh_zb] built (65.1s)
  [final:all] built (67.5s): basket_minute_wide_20260701_bfeca5272447.pkl
[DATE 2/22] 20260702: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260630.csv, components=300
  [partition:sh_kcb] built (28.4s)
  [partition:sz_cyb] built (56.2s)
  [partition:sz_zb] built (59.8s)
  [partition:sh_zb] built (66.6s)
  [final:all] built (68.8s): basket_minute_wide_20260702_ea54ee2142d4.pkl
[DATE 3/22] 20260703: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260630.csv, components=300
  [partition:sh_kcb] built (29.5s)
  [partition:sz_cyb] built (82.2s)
  [partition:sz_zb] built (94.4s)
  [partition:sh_zb] built (120.6s)
  [final:all] built (124.8s): basket_minute_wide_20260703_bfa50f229284.pkl
[DATE 4/22] 20260

In [34]:
generated_cache_paths = generate_csi300_caches(
    trade_dates,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
    max_workers=MAX_WORKERS,
    force_rebuild=FORCE_REBUILD,
)

print(f"Complete caches available after generation: {len(generated_cache_paths)}")
for cache_path in generated_cache_paths:
    print(" ", cache_path)


[DATE 1/22] 20260701: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260630.csv, components=300
  [final:all] cache_hit: basket_minute_wide_20260701_bfeca5272447.pkl
[DATE 2/22] 20260702: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260630.csv, components=300
  [final:all] cache_hit: basket_minute_wide_20260702_ea54ee2142d4.pkl
[DATE 3/22] 20260703: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260630.csv, components=300
  [final:all] cache_hit: basket_minute_wide_20260703_bfa50f229284.pkl
[DATE 4/22] 20260706: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260630.csv, components=300
  [final:all] cache_hit: basket_minute_wide_20260706_b0ca6a949f2e.pkl
[DATE 5/22] 20260707: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260630.csv, components=300
  [final:all] cache_hit: basket_minute_wide_20260707_14ca0077a818.pkl
[DATE 6/22] 20260708: preparing CSI 300 cache with max_workers=4
  weights=

## Merge, validate, and clean partitions

The merge range uses the same `START_DATE` and `END_DATE`. Partition files are deleted only after the complete cache passes schema, date, stock-universe, missing-stock, and minute-row validation.


In [35]:
complete_cache_paths = merge_partition_caches_for_range(
    START_DATE,
    END_DATE,
    xtdata_client=xtdata,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
    delete_partition_caches=DELETE_PARTITION_CACHES,
    overwrite_complete_cache=OVERWRITE_COMPLETE_CACHE,
)

print(f"Validated complete caches: {len(complete_cache_paths)}")
for cache_path in complete_cache_paths:
    print(" ", cache_path)


Merge trading dates (22): 20260701..20260730
[MERGE 1/22] 20260701
  [20260701] complete cache hit: basket_minute_wide_20260701_bfeca5272447.pkl
  [20260701] deleted partition: minute_partition_20260701_sh_kcb_3ffd4116b5c5.pkl
  [20260701] deleted partition: minute_partition_20260701_sh_zb_f21633dda501.pkl
  [20260701] deleted partition: minute_partition_20260701_sz_cyb_dc897d9c6da1.pkl
  [20260701] deleted partition: minute_partition_20260701_sz_zb_4038cd16672f.pkl
[MERGE 2/22] 20260702
  [20260702] complete cache hit: basket_minute_wide_20260702_ea54ee2142d4.pkl
  [20260702] deleted partition: minute_partition_20260702_sh_kcb_6826ff574251.pkl
  [20260702] deleted partition: minute_partition_20260702_sh_zb_40a79375bc7b.pkl
  [20260702] deleted partition: minute_partition_20260702_sz_cyb_1cf4f2030d4e.pkl
  [20260702] deleted partition: minute_partition_20260702_sz_zb_5035434a33fd.pkl
[MERGE 3/22] 20260703
  [20260703] complete cache hit: basket_minute_wide_20260703_bfa50f229284.pkl
  [

In [42]:
# report reader
# 读取 data\000300\_tick_cache_correlation_v03\ 目录下，所有以“unavailable”开头的csv文件
import pandas as pd

unavailable_files = list(CACHE_DIR.glob("unavailable*.csv"))
df = pd.DataFrame()
for file in unavailable_files:
    df = pd.concat([df, pd.read_csv(file)])

df = df.sort_values(by="trade_date").dropna().reset_index(drop=True)[["trade_date", "reason", "message", "errors"]]
df.to_csv(CACHE_DIR / "unavailable_report.csv", index=False)